# 02. Matching monotowns to OKTMO codes

**Goal:** find the official code of each monotown so it can be joined with statistics.

**What is OKTMO?** OKTMO (Russian Classification of Territories of Municipal Formations) is the official ID number of every municipality in Russia. Names of the same place vary between documents, but the code stays the same, so all joins in this project use codes.

The code is hierarchical, from large to small. Example, Slantsy:
- `41` Leningrad Oblast
- `41642000` Slantsy municipal district
- `41642101` Slantsy urban settlement (the town itself)

In [3]:
import pandas as pd
data = "../data/"

mono = pd.read_csv(data + "monotowns_2026.csv", encoding="utf-8-sig")
mono["category"] = pd.cut(mono["number"], bins=[0, 52, 201, 264], labels=[1, 2, 3])

ref = pd.read_csv(data + "processed/population.csv", dtype={"oktmo": str, "oktmo_stable": str})
ref = ref[ref["year"] == 2025]
print(ref.shape)

(13747, 8)


In [5]:
examples = "Райчихинск|Онежский|Воркута|Тольятти|Кондопож|Сланцев|Жешарт|Нерюнгри"
ref[ref["municipality"].str.contains(examples, na=False)][
    ["region_name", "mun_district", "municipality", "mun_type", "oktmo"]]

,region_name,mun_district,municipality,mun_type,oktmo
18544,Амурская область,город Райчихинск,город Райчихинск,"Городской округ, городской округ с внутригород...",10720000
18559,Архангельская область,Онежский муниципальный округ,Онежский муниципальный округ,Муниципальный округ,11546000
22750,Ленинградская область,Сланцевский муниципальный район,Сланцевский муниципальный район,Муниципальный район,41642000
22751,Ленинградская область,Сланцевский муниципальный район,Сланцевское,Городское поселение,41642101
28132,Республика Карелия,Кондопожский муниципальный район,Кондопожский муниципальный район,Муниципальный район,86615000
28133,Республика Карелия,Кондопожский муниципальный район,Кондопожское,Городское поселение,86615101
28218,Республика Коми,Муниципальный округ Воркута,Муниципальный округ Воркута,Муниципальный округ,87510000
28355,Республика Коми,Усть-Вымский муниципальный район,Жешарт,Городское поселение,87644155
30688,Республика Саха (Якутия),Нерюнгринский муниципальный район,Нерюнгринский муниципальный район,Муниципальный район,98660000
30689,Республика Саха (Якутия),Нерюнгринский муниципальный район,Город Нерюнгри,Городское поселение,98660101


In [7]:
missing = sorted(set(mono["region"]) - set(ref["region_name"]))
print(missing)

['Донецкая Народная Республика', 'Запорожская область', 'Кемеровская область - Кузбасс', 'Луганская Народная Республика', 'Республика Бурятия', 'Республика Дагестан', 'Республика Крым']


In [17]:
import re

STOP = {"муниципальное", "образование", "муниципальный", "муниципального", "округ",
        "район", "района", "городское", "городской", "городского", "поселение",
        "город", "поселок", "рабочий", "типа", "закрытое", "административно-территориальное",
        "подведомственной", "территорией", "административным", "центром", "области",
        "края", "республики", "народной", "автономной", "кузбасса", "поссовет", "улус"}

def clean_words(text):
    text = re.sub(r"[^а-я\- ]", " ", str(text).lower().replace("ё", "е"))
    words = [w.strip("-") for w in text.split()]
    return [w for w in words if w not in STOP and len(w) > 2]

def stem(word):
    if "-" in word:                      # Усть-Илимск -> "усть-илим"
        a, b = word.split("-", 1)
        return a + "-" + b[:4]
    return word[:5]                      # Сланцевское -> "сланц"

def mono_type(name):
    n = name.lower()
    if "поселени" in n or "района" in n or "улус" in n:
        return "Городское поселение"
    if "муниципальный округ" in n or "муниципальный район" in n:
        return "Муниципальный округ"
    return "Городской округ, городской округ с внутригородским делением"

UPPER = ["Муниципальный округ", "Муниципальный район",
         "Городской округ, городской округ с внутригородским делением"]

#root of each Rosstat name = root of its longest distinctive word
ref["stem"] = ref["municipality"].apply(
    lambda n: stem(max(clean_words(n), key=len)) if clean_words(n) else None)

results = []
for _, m in mono.iterrows():
    region_stems = {stem(w) for w in clean_words(m["region_ref"])}
    words = [w for w in clean_words(m["municipality"]) if stem(w) not in region_stems]
    text = " ".join(words)
    target = mono_type(m["municipality"])

    cands = ref[(ref["region_name"] == m["region_ref"]) & ref["stem"].notna()].copy()
    allowed = ["Городское поселение"] if target == "Городское поселение" else UPPER
    cands = cands[cands["mun_type"].isin(allowed)]
    cands["pos"] = cands["stem"].apply(lambda s: text.find(s))
    cands = cands[cands["pos"] >= 0]
    cands["type_ok"] = cands["mun_type"] == target
    cands = cands.sort_values(["type_ok", "pos"], ascending=[False, True])

    best = cands.iloc[0] if len(cands) else None
    results.append({
        "number": m["number"],
        "mono_name": m["municipality"],
        "region": m["region"],
        "n_candidates": len(cands),
        "ref_name": best["municipality"] if best is not None else None,
        "ref_type": best["mun_type"] if best is not None else None,
        "oktmo": best["oktmo"] if best is not None else None,
    })

match = pd.DataFrame(results)
print("matched:", match["oktmo"].notna().sum(), "of", len(match))
print("several candidates:", (match["n_candidates"] > 1).sum())
match[match["oktmo"].isna()][["number", "mono_name", "region"]]

matched: 232 of 264
several candidates: 66


,number,mono_name,region
3,4,Муниципальное образование городской округ Енак...,Донецкая Народная Республика
4,5,Городской округ Мариуполь Донецкой Народной Ре...,Донецкая Народная Республика
5,6,Городской округ Торез Донецкой Народной Респуб...,Донецкая Народная Республика
6,7,Шахтерский муниципальный округ Донецкой Народн...,Донецкая Народная Республика
9,10,Городской округ Энергодар Запорожской области,Запорожская область
23,24,Муниципальное образование городской округ горо...,Луганская Народная Республика
24,25,Муниципальное образование городской округ горо...,Луганская Народная Республика
25,26,Муниципальное образование Краснодонский муници...,Луганская Народная Республика
31,32,"Городской округ ""город Дагестанские Огни""",Республика Дагестан
37,38,Муниципальное образование муниципальный округ ...,Республика Крым


In [11]:
print(sorted(set(mono["region"]) - set(ref["region_name"])))
print()
print([r for r in ref["region_name"].unique() if any(x in r for x in ["Кемер", "Самар", "Крым"])])

['Донецкая Народная Республика', 'Запорожская область', 'Кемеровская область - Кузбасс', 'Луганская Народная Республика', 'Республика Бурятия', 'Республика Дагестан', 'Республика Крым']

['Кемеровская область', 'Самарская область']


In [13]:
names = ["Коряжм", "Новодвинск", "Огни", "Каспийск", "Болхов", "Колпн", "Сердобск",
         "Заречн", "Каменск", "Селенгин", "Мокшан", "Кимовск", "Белев", "Первомайск",
         "Лесной", "Северск", "Озерск", "Снежинск", "Трехгорн", "Железногорск"]
ref[ref["municipality"].str.contains("|".join(names), case=False, na=False)][
    ["region_name", "mun_district", "municipality", "mun_type", "oktmo"]]

,region_name,mun_district,municipality,mun_type,oktmo
17016,Алтайский край,Алтайский муниципальный район,Нижнекаменский сельсовет,Сельское поселение,01602446
17036,Алтайский край,Бийский муниципальный район,Лесной сельсовет,Сельское поселение,01604437
17040,Алтайский край,Бийский муниципальный район,Первомайский сельсовет,Сельское поселение,01604470
17093,Алтайский край,Егорьевский муниципальный район,Первомайский сельсовет,Сельское поселение,01609444
17101,Алтайский край,Ельцовский муниципальный район,Новокаменский сельсовет,Сельское поселение,01610444
...,...,...,...,...,...
29872,Республика Татарстан,Черемшанский муниципальный район,Нижнекаменское,Сельское поселение,92658438
29963,Удмуртская Республика,Муниципальный округ Юкаменский район,Муниципальный округ Юкаменский район,Муниципальный округ,94548000
30033,Чеченская Республика,Грозненский муниципальный район,Первомайское,Сельское поселение,96607431
30548,Республика Саха (Якутия),Олекминский муниципальный район,Поселок Заречный,Городское поселение,98641493


In [15]:
mono["region_ref"] = mono["region"].replace({"Кемеровская область - Кузбасс": "Кемеровская область"})
print([r for r in ref["region_name"].unique() if any(x in r for x in ["Бурят", "Дагест", "Крым"])])

[]


In [19]:
NEW = ["Донецкая Народная Республика", "Луганская Народная Республика", "Запорожская область"]
todo = match[match["oktmo"].isna() & ~match["region"].isin(NEW)]

for _, t in todo.iterrows():
    reg = mono.loc[mono["number"] == t["number"], "region_ref"].iloc[0]
    region_stems = {stem(w) for w in clean_words(reg)}
    keys = [w[:4] for w in clean_words(t["mono_name"]) if stem(w) not in region_stems]
    hits = ref[(ref["region_name"] == reg)
               & ref["mun_type"].isin(UPPER + ["Городское поселение"])
               & ref["municipality"].str.lower().str.contains("|".join(keys), na=False)]
    print(t["number"], t["mono_name"][:70])
    if len(hits) == 0:
        print("     no candidates")
    for _, h in hits.iterrows():
        print("    ", h["municipality"], "|", h["mun_type"], "|", h["oktmo"])

32 Городской округ "город Дагестанские Огни"
     no candidates
38 Муниципальное образование муниципальный округ город Армянск Республики
     no candidates
45 Городской округ Тольятти Самарской области
     no candidates
55 Городской округ Архангельской области "Город Коряжма"
     no candidates
56 Городской округ Архангельской области "Город Новодвинск"
     no candidates
103 Городской округ "Закрытое административно-территориальное образование 
     no candidates
123 Городское поселение город Болхов Болховского муниципального района Орл
     Болховский муниципальный район | Муниципальный район | 54604000
125 Муниципальное образование городское поселение - поселок городского тип
     Колпнянский муниципальный район | Муниципальный район | 54623000
126 Городское поселение город Сердобск муниципального района Сердобский ра
     Малосердобинский муниципальный район | Муниципальный район | 56644000
     Сердобский муниципальный район | Муниципальный район | 56656000
127 Городской округ г